In [ ]:
import pandas as pd
import numpy as np
import random
import scipy

np.float = float


from sklearn.neighbors import NearestNeighbors
from scipy.sparse import csr_matrix

from evidently.calculations.stattests import wasserstein_stat_test
from evidently.calculations.stattests import ks_stat_test
from evidently.calculations.stattests.jensenshannon import jensenshannon_stat_test
from evidently.calculations.stattests.kl_div import kl_div_stat_test
from evidently.calculations.stattests import hellinger_stat_test
from evidently.core import ColumnType

In [ ]:
reviews = pd.read_json("Amazon_Fashion.jsonl", lines=True)

In [ ]:
reviews = reviews[reviews["rating"] != 3]

In [ ]:
def create_matrix(df):
    N = len(df["user_id"].unique())
    M = len(df["product_id"].unique())

    # Map Ids to indices
    user_mapper = dict(zip(np.unique(df["user_id"]), list(range(N))))
    product_mapper = dict(zip(np.unique(df["product_id"]), list(range(M))))

    # Map indices to IDs
    user_inv_mapper = dict(zip(list(range(N)), np.unique(df["user_id"])))
    product_inv_mapper = dict(zip(list(range(M)), np.unique(df["product_id"])))

    user_index = [user_mapper[i] for i in df["user_id"]]
    product_index = [product_mapper[i] for i in df["product_id"]]

    X = csr_matrix((df["rating"], (product_index, user_index)), shape=(M, N))

    return X, user_mapper, product_mapper, user_inv_mapper, product_inv_mapper

In [ ]:
reviews["product_id"] = reviews["parent_asin"]

In [ ]:
X, user_mapper, product_mapper, user_inv_mapper, product_inv_mapper = create_matrix(
    reviews
)

In [ ]:
kNN = NearestNeighbors(n_neighbors=5, algorithm="brute", metric="cosine")
kNN.fit(X)

In [ ]:
def find_similar_products(product_id, X, k, metric="cosine", show_distance=False):
    neighbour_ids = []

    product_ind = product_mapper[product_id]
    product_vec = X[product_ind]
    k += 1
    product_vec = product_vec.reshape(1, -1)
    neighbour = kNN.kneighbors(product_vec, return_distance=show_distance)
    for i in range(0, k):
        n = neighbour.item(i)
        neighbour_ids.append(product_inv_mapper[n])
    neighbour_ids.pop(0)
    return neighbour_ids

In [ ]:
find_similar_products("B00LOPVX74", X, 4)

In [ ]:
import time

In [ ]:
product_list = list(product_inv_mapper.values())

In [ ]:
random.sample(product_list, 1)

In [ ]:
time_list = []
for i in range(100):
    prod = random.sample(product_list, 1)[0]
    time_start = time.perf_counter()
    find_similar_products(prod, X, 4)
    time_elapsed = time.perf_counter() - time_start
    time_list.append(time_elapsed)
print(np.mean(time_list))
print(np.min(time_list))
print(np.max(time_list))

In [ ]:
np.max(time_list) * 1000

In [ ]:
print(str(time_elapsed * 1000) + " ms")

In [ ]:
pd.options.mode.copy_on_write = True
data = pd.read_json("Amazon_Fashion.jsonl", lines=True)

In [ ]:
reference_data = data[data["timestamp"] > "2015-01-01"]
reference_data = reference_data[reference_data["timestamp"] < "2020-01-01"]

evaluation_data = data[data["timestamp"] >= "2020-01-01"]
evaluation_data = evaluation_data.set_index(evaluation_data["timestamp"])
evaluation_data.sort_index(inplace=True)

reference_array = reference_data["rating"].values

In [ ]:
def get_sliding_window_with_newvals(x):
    windows = []
    times = []
    new_values = []
    start_time = x["timestamp"].values[0]
    day = np.timedelta64(1, "D")
    week = np.timedelta64(30, "D")
    end_time = start_time + week
    while end_time <= x["timestamp"].values[-1]:
        end_time = start_time + week
        window = x[x["timestamp"] >= start_time]
        new_values.append(
            window[window["timestamp"] < (start_time + day)]["rating"].values
        )
        window = window[window["timestamp"] < end_time]
        windows.append(window["rating"].values)
        times.append(end_time)
        start_time = start_time + day

    return times, windows, new_values

In [ ]:
sliding_windows = get_sliding_window_with_newvals(evaluation_data)

In [ ]:
window_drift_wasserstein = scipy.stats.wasserstein_distance(
    reference_array, sliding_windows[1][30]
)

In [ ]:
time_list_with_drift = []
for i in range(100):
    prod = random.sample(product_list, 1)[0]
    time_start = time.perf_counter()
    find_similar_products(prod, X, 4)
    window_drift_wasserstein = wasserstein_stat_test(
        pd.Series(reference_array),
        pd.Series(sliding_windows[1][30 + i]),
        ColumnType.Numerical,
        0.1,
    )
    time_elapsed = time.perf_counter() - time_start
    time_list_with_drift.append(time_elapsed)
print(np.mean(time_list_with_drift))
print(np.min(time_list_with_drift))
print(np.max(time_list_with_drift))

In [ ]:
time_list_with_drift = []
for i in range(100):
    prod = random.sample(product_list, 1)[0]
    time_start = time.perf_counter()
    find_similar_products(prod, X, 4)
    window_drift_wasserstein = ks_stat_test(
        pd.Series(reference_array),
        pd.Series(sliding_windows[1][30 + i]),
        ColumnType.Numerical,
        0.1,
    )
    time_elapsed = time.perf_counter() - time_start
    time_list_with_drift.append(time_elapsed)
print(np.mean(time_list_with_drift))
print(np.min(time_list_with_drift))
print(np.max(time_list_with_drift))

In [ ]:
time_list_with_drift = []
for i in range(100):
    prod = random.sample(product_list, 1)[0]
    time_start = time.perf_counter()
    find_similar_products(prod, X, 4)
    window_drift_wasserstein = scipy.stats.energy_distance(
        reference_array, sliding_windows[1][30 + i]
    )
    time_elapsed = time.perf_counter() - time_start
    time_list_with_drift.append(time_elapsed)
print(np.mean(time_list_with_drift))
print(np.min(time_list_with_drift))
print(np.max(time_list_with_drift))

In [ ]:
time_list_with_drift = []
for i in range(100):
    prod = random.sample(product_list, 1)[0]
    time_start = time.perf_counter()
    find_similar_products(prod, X, 4)
    window_drift_wasserstein = jensenshannon_stat_test(
        pd.Series(reference_array),
        pd.Series(sliding_windows[1][30 + i]),
        ColumnType.Numerical,
        0.1,
    )
    time_elapsed = time.perf_counter() - time_start
    time_list_with_drift.append(time_elapsed)
print(np.mean(time_list_with_drift))
print(np.min(time_list_with_drift))
print(np.max(time_list_with_drift))

In [ ]:
time_list_with_drift = []
for i in range(100):
    prod = random.sample(product_list, 1)[0]
    time_start = time.perf_counter()
    find_similar_products(prod, X, 4)
    window_drift_wasserstein = kl_div_stat_test(
        pd.Series(reference_array),
        pd.Series(sliding_windows[1][30 + i]),
        ColumnType.Numerical,
        0.1,
    )
    time_elapsed = time.perf_counter() - time_start
    time_list_with_drift.append(time_elapsed)
print(np.mean(time_list_with_drift))
print(np.min(time_list_with_drift))
print(np.max(time_list_with_drift))

In [ ]:
time_list_with_drift = []
for i in range(100):
    prod = random.sample(product_list, 1)[0]
    time_start = time.perf_counter()
    find_similar_products(prod, X, 4)
    window_drift_wasserstein = hellinger_stat_test(
        pd.Series(reference_array),
        pd.Series(sliding_windows[1][30 + i]),
        ColumnType.Numerical,
        0.1,
    )
    time_elapsed = time.perf_counter() - time_start
    time_list_with_drift.append(time_elapsed)
print(np.mean(time_list_with_drift))
print(np.min(time_list_with_drift))
print(np.max(time_list_with_drift))